In [ ]:
import sys
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import geopandas as gpd
from shapely.geometry import mapping
from scipy.stats import spearmanr, pearsonr
import ts_onset_cess as ocd
import pandas as pd
from fapar_def import fapar_read

import warnings
warnings.filterwarnings('ignore')

In [ ]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"

### Read in calculated recycling to add to the precipitation plots

In [ ]:
Y1=1990
Y2=2024

datao ="/Volumes/ESA_F4R/ed_prepare/2026_rho/mint_r02_it1000_tole3/bands_rho/" 
S_NAME = "S_SE" # S_SE or S_LSE 
L_NAME = "L_M" # L_M or L_HI
band = {'N':[5,12,10,31],'EQ':[-5,5,8,29],'S':[-15,-5,12,31]}

B = 'EQ'

### Read in ERA5 for a band 


In [ ]:
Y1=1981
Y2=2025
for Y in range(Y1,Y2+1):
    prp = xr.open_mfdataset('/Volumes/blue_wd/chirps_daily/chirps-v2.0.'+str(Y)+'.days_p05.nc')['precip']
    prp = prp.rename({'latitude':'lat','longitude':'lon'})
    prpN = prp.sel(lat=slice(5,12),lon=slice(10,31),drop=True).load()
    prpEQ = prp.sel(lat=slice(-5,5),lon=slice(8,29),drop=True).load()
    prpS = prp.sel(lat=slice(-15,-5),lon=slice(12,31),drop=True).load()
    if Y==Y1:
        prpaN = prpN
        prpaEQ = prpEQ
        prpaS = prpS
    else:
        prpaN = xr.concat([prpaN,prpN],dim='time')
        prpaEQ = xr.concat([prpaEQ,prpEQ],dim='time')
        prpaS = xr.concat([prpaS,prpS],dim='time')
        prpN.close()
        prpEQ.close()
        prpS.close()
prpaN.to_netcdf('/Users/ellendyer/Documents/GitHub/Isotopes_F4R/data/chirps_N.nc',engine='h5netcdf')
prpaN.close()
prpaEQ.to_netcdf('/Users/ellendyer/Documents/GitHub/Isotopes_F4R/data/chirps_EQ.nc',engine='h5netcdf')
prpaEQ.close()
prpaS.to_netcdf('/Users/ellendyer/Documents/GitHub/Isotopes_F4R/data/chirps_S.nc',engine='h5netcdf')
prpaS.close()
        

## Equatorial Band

- Checking to see if wet and dry seasons have consistently different onset and cessation dates

- Calculate wet and dry season composites for season 1 and season 2

- Calculate the onset and cessation of full climatology for all seasons

- Calculate onset and cessation of wet and dry composite seasons

- Calculate mean of onset and cessation composites

- Plot everything

In [ ]:
#Equatorial Band

P = pr['EQ'] 
FP = fapar.sel(lat=slice(-5,5),lon=slice(8,29)).mean(dim=('lat','lon'))
FP = FP.groupby('time.dayofyear').mean('time')

#Composite season years - S1
PA = P.mean(dim=('lat','lon')).where(P['time.dayofyear']<200).groupby('time.year').mean('time')
sort_index = PA.sortby(PA)
dryS1 = sort_index[0:5]['year']
wetS1 = sort_index[-5:]['year']
print('Dry years S1',dryS1['year'].values)
print('Wet years S1',wetS1['year'].values)
PW1 = P.sel(time=P['time.year'].isin(wetS1['year'].values)).groupby('time.dayofyear').mean(dim=('time','lat','lon'))
PD1 = P.sel(time=P['time.year'].isin(dryS1['year'].values)).groupby('time.dayofyear').mean(dim=('time','lat','lon'))
#Composite season years - S2
PA = P.mean(dim=('lat','lon')).where(P['time.dayofyear']>200).groupby('time.year').mean('time')
sort_index = PA.sortby(PA)
dryS2 = sort_index[0:5]['year']
wetS2 = sort_index[-5:]['year']
print('Dry years S2',dryS2['year'].values)
print('Wet years S2',wetS2['year'].values)
PW2 = P.sel(time=P['time.year'].isin(wetS2['year'].values)).groupby('time.dayofyear').mean(dim=('time','lat','lon'))
PD2 = P.sel(time=P['time.year'].isin(dryS2['year'].values)).groupby('time.dayofyear').mean(dim=('time','lat','lon'))
print('----------------------------------------------------')


#Season timing
power_ratio,on1,ce1,on2,ce2,on3,ce3,on1_years,ce1_years,on2_years,ce2_years,on3_years,ce3_years = ocd.xarray_on_cess_point(P.mean(dim=("lat","lon")))

mean_onset_1 = on1_years.mean('year').values 
mean_cess_1 = ce1_years.mean('year').values 
mean_onset_2 = on2_years.mean('year').values 
mean_cess_2 = ce2_years.mean('year').values
print('All years mean onset 1', mean_onset_1, 'All years mean cessation 1', mean_cess_1)
print('All years mean onset 2', mean_onset_2, 'All years mean cessation 2', mean_cess_2)
print('----------------------------------------------------')

#Onset and cessation of wet and dry composite years

print('Wet years onset 1: ',on1_years.sel(year=on1_years['year'].isin(wetS1['year'].values)).mean('year').values ,
      'Dry years onset 1: ',on1_years.sel(year=on1_years['year'].isin(dryS1['year'].values)).mean('year').values)
print('Wet years cessation 1: ',ce1_years.sel(year=ce1_years['year'].isin(wetS1['year'].values)).mean('year').values ,
      'Dry years cessation1: ',ce1_years.sel(year=ce1_years['year'].isin(dryS1['year'].values)).mean('year').values)

print('Wet years onset 2: ',on2_years.sel(year=on2_years['year'].isin(wetS2['year'].values)).mean('year').values ,
      'Dry years onset 2: ',on2_years.sel(year=on2_years['year'].isin(dryS2['year'].values)).mean('year').values)
print('Wet years cessation 2: ',ce2_years.sel(year=ce2_years['year'].isin(wetS2['year'].values)).mean('year').values ,
      'Dry years cessation2: ',ce2_years.sel(year=ce2_years['year'].isin(dryS2['year'].values)).mean('year').values)
print('----------------------------------------------------')

#Onset and cessation of early and late composite years

sort_index_onset = on1_years.sortby(on1_years)
early_onset1 = sort_index_onset[0:5]
late_onset1 = sort_index_onset[-5:]
early_onset_precip = PA.where(PA['year'].isin(early_onset1['year']),drop=True)
late_onset_precip = PA.where(PA['year'].isin(late_onset1['year']),drop=True)
print('Mean early onset comp S1 ',early_onset1.mean('year').values, 'Years: ',early_onset1['year'].values, 'Early onset comp S1 ',early_onset1.values)
print('Mean late onset comp S1 ',late_onset1.mean('year').values, 'Years: ',late_onset1['year'].values, 'Late onset comp S1 ',late_onset1.values)

sort_index_cess = ce1_years.sortby(ce1_years)
early_cess1 = sort_index_cess[0:5]
late_cess1 = sort_index_cess[-5:]
early_cess_precip = PA.where(PA['year'].isin(early_cess1['year']),drop=True)
late_cess_precip = PA.where(PA['year'].isin(late_cess1['year']),drop=True)
print('Mean early cess comp S1 ',early_cess1.mean('year').values, 'Years: ',early_cess1['year'].values, 'Early cess comp S1 ',early_cess1.values)
print('Mean late cess comp S1 ',late_cess1.mean('year').values, 'Years: ',late_cess1['year'].values, 'Late cess comp S1 ',late_cess1.values)
print('----------------------------------------------------')

sort_index_onset = on2_years.sortby(on2_years)
early_onset2 = sort_index_onset[0:5]
late_onset2 = sort_index_onset[-5:]
early_onset_precip = PA.where(PA['year'].isin(early_onset2['year']),drop=True)
late_onset_precip = PA.where(PA['year'].isin(late_onset2['year']),drop=True)
print('Mean early onset comp S2 ',early_onset2.mean('year').values, 'Years: ',early_onset2['year'].values, 'Early onset comp S2 ',early_onset2.values)
print('Mean late onset comp S2 ',late_onset2.mean('year').values, 'Years: ',late_onset2['year'].values, 'Late onset comp S2 ',late_onset2.values)

sort_index_cess = ce2_years.sortby(ce2_years)
early_cess2 = sort_index_cess[0:5]
late_cess2 = sort_index_cess[-5:]
early_cess_precip = PA.where(PA['year'].isin(early_cess2['year']),drop=True)
late_cess_precip = PA.where(PA['year'].isin(late_cess2['year']),drop=True)
print('Mean early cess comp S2 ',early_cess2.mean('year').values, 'Years: ',early_cess2['year'].values, 'Early cess comp S2 ',early_cess2.values)
print('Mean late cess comp S2 ',late_cess2.mean('year').values, 'Years: ',late_cess2['year'].values, 'Late cess comp S2 ',late_cess2.values)
print('----------------------------------------------------')


#Plot everything - SEASON 1

fig,ax = plt.subplots(figsize=(13, 6))
Pdays = P.mean(dim = ('lat','lon')).groupby('time.dayofyear').mean(dim = ('time'))
Pplot = Pdays.rolling(dayofyear=20, center=True).mean('dayofyear')
Pplot.plot(linewidth=3,label='Precip Climatology')

plt.axvline(x=on1_years.mean('year').values, color='firebrick', linestyle='-', linewidth=5,alpha=0.5,label='Mean Onset')
plt.axvline(x=early_onset1.mean('year'), color='firebrick', linestyle='-.', linewidth=2,alpha=0.5,label='Early Onset')
plt.axvline(x=late_onset1.mean('year'), color='firebrick', linestyle='--', linewidth=2,alpha=0.5,label='Late Onset')

plt.axvline(x=ce1_years.mean('year').values, color='green', linestyle='-', linewidth=5,alpha=0.5,label='Mean Cessation')
plt.axvline(x=early_cess1.mean('year'), color='green', linestyle='-.', linewidth=2,alpha=0.5,label='Early Cessation')
plt.axvline(x=late_cess1.mean('year'), color='green', linestyle='--', linewidth=2,alpha=0.5,label='Late Cessation')

plt.axvline(x=on1_years.sel(year=on1_years['year'].isin(wetS1['year'].values)).mean('year').values, color='purple', linestyle='--', linewidth=2,alpha=1,label='Wet years Mean Onset')
plt.axvline(x=on1_years.sel(year=on1_years['year'].isin(dryS1['year'].values)).mean('year').values, color='orange', linestyle='--', linewidth=2,alpha=1,label='Dry years Mean Onset')
plt.axvline(x=ce1_years.sel(year=ce1_years['year'].isin(wetS1['year'].values)).mean('year').values, color='purple', linestyle='dotted', linewidth=2,alpha=1,label='Wet years Mean Cessation')
plt.axvline(x=ce1_years.sel(year=ce1_years['year'].isin(dryS1['year'].values)).mean('year').values, color='orange', linestyle='dotted', linewidth=2,alpha=1,label='Dry years Mean Cessation')

PonE = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(early_onset1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PonE = PonE.where(PonE['dayofyear']<200)
PonL = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(late_onset1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PonL = PonL.where(PonL['dayofyear']<200)

PceE = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(early_cess1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PceE = PceE.where(PceE['dayofyear']<200)
PceL = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(late_cess1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PceL = PceL.where(PceL['dayofyear']<200)

#plt.fill_between(PonL[0:365].dayofyear,PonE[0:365],PonL[0:365], color='red',alpha=0.3,label="Early vs Late Onset")
#plt.fill_between(PceL[0:365].dayofyear,PceE[0:365],PceL[0:365],color='green',alpha=0.3,label="Early vs Late Cess")
#PonE.plot(linestyle='--',color='red',alpha=1,label="Early Onset Clim")
#PceL.plot(linestyle='--',color='green',alpha=1,label="Late Cessation Clim")

PW1 = PW1.rolling(dayofyear=20, center=True).mean('dayofyear').where(PW1['dayofyear']<200)
PD1 = PD1.rolling(dayofyear=20, center=True).mean('dayofyear').where(PW1['dayofyear']<200)
PW1.plot(linestyle='-',color='purple',alpha=1,label="Wet Composite Clim")
PD1.plot(linestyle='-',color='orange',alpha=1,label="Dry Composite Clim")

ax2 = ax.twinx()
ax3 = ax.twinx()
ax3.spines.right.set_position(("axes", 1.08))

ax2.plot(rho_mon['EQ']['dayofyear'],rho_mon['EQ'],linestyle='-',linewidth=2,color='darkgreen',alpha=0.3)
ax2.scatter(rho_mon['EQ']['dayofyear'],rho_mon['EQ'],s=70,color='darkgreen',label='Recycling Ratio',alpha=0.8)
ax2.set(ylim=(0,0.4), ylabel='Recycling rho')

FPR = FP.rolling(dayofyear=3, center=True).mean('dayofyear')
ax3.plot(FPR['dayofyear'],FPR,linestyle='-',linewidth=3,color='teal',alpha=0.7,label='FAPAR')
ax3.set(ylim=(0.6,0.85),ylabel='Fraction of Absorbed \n Photosynthetically Active Radiation')

ax.legend(loc='best')
ax2.legend(loc=1)
ax3.legend(loc=2)

plt.title('Equatorial Band - Season 1')
plt.savefig(datap+'EQ_S1_timing_pres.png',dpi=200,transparent=True,bbox_inches='tight')
plt.show()
plt.clf()
plt.close()


#Plot everything - SEASON 2

fig,ax = plt.subplots(figsize=(13, 6))
Pdays = P.mean(dim = ('lat','lon')).groupby('time.dayofyear').mean(dim = ('time'))
Pplot = Pdays.rolling(dayofyear=20, center=True).mean('dayofyear')
Pplot.plot(linewidth=3,label='Precip Climatology')

plt.axvline(x=on2_years.mean('year').values, color='firebrick', linestyle='-', linewidth=5,alpha=0.5,label='Mean Onset')
plt.axvline(x=early_onset2.mean('year'), color='firebrick', linestyle='-.', linewidth=2,alpha=0.5,label='Early Onset')
plt.axvline(x=late_onset2.mean('year'), color='firebrick', linestyle='--', linewidth=2,alpha=0.5,label='Late Onset')

plt.axvline(x=ce2_years.mean('year').values, color='green', linestyle='-', linewidth=5,alpha=0.5,label='Mean Cessation')
plt.axvline(x=early_cess2.mean('year'), color='green', linestyle='-.', linewidth=2,alpha=0.5,label='Early Cessation')
plt.axvline(x=late_cess2.mean('year'), color='green', linestyle='--', linewidth=2,alpha=0.5,label='Late Cessation')

plt.axvline(x=on2_years.sel(year=on2_years['year'].isin(wetS2['year'].values)).mean('year').values, color='purple', linestyle='--', linewidth=2,alpha=1,label='Wet years Mean Onset')
plt.axvline(x=on2_years.sel(year=on2_years['year'].isin(dryS2['year'].values)).mean('year').values, color='orange', linestyle='--', linewidth=2,alpha=1,label='Dry years Mean Onset')
plt.axvline(x=ce2_years.sel(year=ce2_years['year'].isin(wetS2['year'].values)).mean('year').values, color='purple', linestyle='dotted', linewidth=2,alpha=1,label='Wet years Mean Cessation')
plt.axvline(x=ce2_years.sel(year=ce2_years['year'].isin(dryS2['year'].values)).mean('year').values, color='orange', linestyle='dotted', linewidth=2,alpha=1,label='Dry years Mean Cessation')

PonE = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(early_onset2['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PonE = PonE.where(PonE['dayofyear']>200)
PonL = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(late_onset2['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PonL = PonL.where(PonL['dayofyear']>200)

PceE = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(early_cess2['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PceE = PceE.where(PceE['dayofyear']>200)
PceL = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(late_cess2['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PceL = PceL.where(PceL['dayofyear']>200)

#plt.fill_between(PonL[0:365].dayofyear,PonE[0:365],PonL[0:365], color='red',alpha=0.3,label="Early vs Late Onset")
#plt.fill_between(PceL[0:365].dayofyear,PceE[0:365],PceL[0:365],color='green',alpha=0.3,label="Early vs Late Cess")
#PonE.plot(linestyle='--',color='red',alpha=1,label="Early Onset Clim")
#PceL.plot(linestyle='--',color='green',alpha=1,label="Late Cessation Clim")

PW2 = PW2.rolling(dayofyear=20, center=True).mean('dayofyear').where(PW2['dayofyear']>200)
PD2 = PD2.rolling(dayofyear=20, center=True).mean('dayofyear').where(PW2['dayofyear']>200)
PW2.plot(linestyle='-',color='purple',alpha=1,label="Wet Composite Clim")
PD2.plot(linestyle='-',color='orange',alpha=1,label="Dry Composite Clim")

ax2 = ax.twinx()
ax3 = ax.twinx()
ax3.spines.right.set_position(("axes", 1.08))

ax2.plot(rho_mon['EQ']['dayofyear'],rho_mon['EQ'],linestyle='-',linewidth=2,color='darkgreen',alpha=0.3)
ax2.scatter(rho_mon['EQ']['dayofyear'],rho_mon['EQ'],s=70,color='darkgreen',label='Recycling Ratio',alpha=0.8)
ax2.set(ylim=(0,0.4),ylabel='Recycling rho')

FPR = FP.rolling(dayofyear=3, center=True).mean('dayofyear')
ax3.plot(FPR['dayofyear'],FPR,linestyle='-',linewidth=3,color='teal',alpha=0.7,label='FAPAR')
ax3.set(ylim=(0.6,0.85),ylabel='Fraction of Absorbed \n Photosynthetically Active Radiation')

ax.legend(loc='best')
ax2.legend(loc=1)
ax3.legend(loc='lower right')

plt.title('Equatorial Band - Season 2')
plt.savefig(datap+'EQ_S2_timing_pres.png',dpi=200,transparent=True,bbox_inches='tight')
plt.show()
plt.clf()
plt.close()


In [ ]:
#North Band

P = pr['N'] 
FP = fapar.sel(lat=slice(5,12),lon=slice(10,31)).mean(dim=('lat','lon'))
FP = FP.groupby('time.dayofyear').mean('time')

#Composite season years - S1
PA = P.mean(dim=('lat','lon')).groupby('time.year').mean('time')
sort_index = PA.sortby(PA)
dryS1 = sort_index[0:5]['year']
wetS1 = sort_index[-5:]['year']
print('Dry years S1',dryS1['year'].values)
print('Wet years S1',wetS1['year'].values)
PW1 = P.sel(time=P['time.year'].isin(wetS1['year'].values)).groupby('time.dayofyear').mean(dim=('time','lat','lon'))
PD1 = P.sel(time=P['time.year'].isin(dryS1['year'].values)).groupby('time.dayofyear').mean(dim=('time','lat','lon'))
print('----------------------------------------------------')


#Season timing
power_ratio,on1,ce1,on2,ce2,on3,ce3,on1_years,ce1_years,on2_years,ce2_years,on3_years,ce3_years = ocd.xarray_on_cess_point(P.mean(dim=("lat","lon")))

mean_onset_1 = on1_years.mean('year').values 
mean_cess_1 = ce1_years.mean('year').values 
mean_onset_2 = on2_years.mean('year').values 
mean_cess_2 = ce2_years.mean('year').values
print('All years mean onset 1', mean_onset_1, 'All years mean cessation 1', mean_cess_1)
print('All years mean onset 2', mean_onset_2, 'All years mean cessation 2', mean_cess_2)
print('----------------------------------------------------')

#Onset and cessation of wet and dry composite years

print('Wet years onset 1: ',on1_years.sel(year=on1_years['year'].isin(wetS1['year'].values)).mean('year').values ,
      'Dry years onset 1: ',on1_years.sel(year=on1_years['year'].isin(dryS1['year'].values)).mean('year').values)
print('Wet years cessation 1: ',ce1_years.sel(year=ce1_years['year'].isin(wetS1['year'].values)).mean('year').values ,
      'Dry years cessation1: ',ce1_years.sel(year=ce1_years['year'].isin(dryS1['year'].values)).mean('year').values)
print('----------------------------------------------------')

#Onset and cessation of early and late composite years

sort_index_onset = on1_years.sortby(on1_years)
early_onset1 = sort_index_onset[0:5]
late_onset1 = sort_index_onset[-5:]
early_onset_precip = PA.where(PA['year'].isin(early_onset1['year']),drop=True)
late_onset_precip = PA.where(PA['year'].isin(late_onset1['year']),drop=True)
print('Mean early onset comp S1 ',early_onset1.mean('year').values, 'Years: ',early_onset1['year'].values, 'Early onset comp S1 ',early_onset1.values)
print('Mean late onset comp S1 ',late_onset1.mean('year').values, 'Years: ',late_onset1['year'].values, 'Late onset comp S1 ',late_onset1.values)

sort_index_cess = ce1_years.sortby(ce1_years)
early_cess1 = sort_index_cess[0:5]
late_cess1 = sort_index_cess[-5:]
early_cess_precip = PA.where(PA['year'].isin(early_cess1['year']),drop=True)
late_cess_precip = PA.where(PA['year'].isin(late_cess1['year']),drop=True)
print('Mean early cess comp S1 ',early_cess1.mean('year').values, 'Years: ',early_cess1['year'].values, 'Early cess comp S1 ',early_cess1.values)
print('Mean late cess comp S1 ',late_cess1.mean('year').values, 'Years: ',late_cess1['year'].values, 'Late cess comp S1 ',late_cess1.values)
print('----------------------------------------------------')

#Plot everything - SEASON 1

fig,ax = plt.subplots(figsize=(13, 6))
Pdays = P.mean(dim = ('lat','lon')).groupby('time.dayofyear').mean(dim = ('time'))
Pplot = Pdays.rolling(dayofyear=20, center=True).mean('dayofyear')
Pplot.plot(linewidth=3,label='Precip Climatology')

plt.axvline(x=on1_years.mean('year').values, color='firebrick', linestyle='-', linewidth=5,alpha=0.5,label='Mean Onset')
plt.axvline(x=early_onset1.mean('year'), color='firebrick', linestyle='-.', linewidth=2,alpha=0.5,label='Early Onset')
plt.axvline(x=late_onset1.mean('year'), color='firebrick', linestyle='--', linewidth=2,alpha=0.5,label='Late Onset')

plt.axvline(x=ce1_years.mean('year').values, color='green', linestyle='-', linewidth=5,alpha=0.5,label='Mean Cessation')
plt.axvline(x=early_cess1.mean('year'), color='green', linestyle='-.', linewidth=2,alpha=0.5,label='Early Cessation')
plt.axvline(x=late_cess1.mean('year'), color='green', linestyle='--', linewidth=2,alpha=0.5,label='Late Cessation')

plt.axvline(x=on1_years.sel(year=on1_years['year'].isin(wetS1['year'].values)).mean('year').values, color='purple', linestyle='--', linewidth=2,alpha=1,label='Wet years Mean Onset')
plt.axvline(x=on1_years.sel(year=on1_years['year'].isin(dryS1['year'].values)).mean('year').values, color='orange', linestyle='--', linewidth=2,alpha=1,label='Dry years Mean Onset')
plt.axvline(x=ce1_years.sel(year=ce1_years['year'].isin(wetS1['year'].values)).mean('year').values, color='purple', linestyle='dotted', linewidth=2,alpha=1,label='Wet years Mean Cessation')
plt.axvline(x=ce1_years.sel(year=ce1_years['year'].isin(dryS1['year'].values)).mean('year').values, color='orange', linestyle='dotted', linewidth=2,alpha=1,label='Dry years Mean Cessation')

PonE = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(early_onset1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PonL = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(late_onset1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')

PceE = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(early_cess1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PceL = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(late_cess1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')

#plt.fill_between(PonE[0:365].dayofyear,PonE[0:365],PonL[0:365], color='red',alpha=0.3,label="Early vs Late Onset")
#plt.fill_between(PceL[0:365].dayofyear,PceE[0:365],PceL[0:365],color='green',alpha=0.3,label="Early vs Late Cess")
#PonE.plot(linestyle='--',color='red',alpha=1,label="Early Onset Clim")
#PceL.plot(linestyle='--',color='green',alpha=1,label="Late Cessation Clim")

PW1 = PW1.rolling(dayofyear=20, center=True).mean('dayofyear')
PD1 = PD1.rolling(dayofyear=20, center=True).mean('dayofyear')
PW1.plot(linestyle='-',color='purple',alpha=1,label="Wet Composite Clim")
PD1.plot(linestyle='-',color='orange',alpha=1,label="Dry Composite Clim")

ax2 = ax.twinx()
ax3 = ax.twinx()
ax3.spines.right.set_position(("axes", 1.08))

ax2.plot(rho_mon['N']['dayofyear'],rho_mon['N'],linestyle='-',linewidth=2,color='darkgreen',alpha=0.3)
ax2.scatter(rho_mon['N']['dayofyear'],rho_mon['N'],s=70,color='darkgreen',label='Recycling Ratio',alpha=0.8)
ax2.set(ylim=(0,0.4),ylabel='Recycling rho')

FPR = FP.rolling(dayofyear=3, center=True).mean('dayofyear')
ax3.plot(FPR['dayofyear'],FPR,linestyle='-',linewidth=3,color='teal',alpha=0.7,label='FAPAR')
ax3.set(ylim=(0,0.85),ylabel='Fraction of Absorbed \n Photosynthetically Active Radiation')

ax.legend(loc='best')
ax2.legend(loc=1)
ax3.legend(loc='lower right')

plt.title('Northern Band')
plt.savefig(datap+'N_S1_timing_pres.png',dpi=200,transparent=True,bbox_inches='tight')
plt.show()
plt.clf()


In [ ]:
#Southern Band

P = pr['S'] 
FP = fapar.sel(lat=slice(-15,-5),lon=slice(12,31)).mean(dim=('lat','lon'))
FP = FP.groupby('time.dayofyear').mean('time')

#Composite season years - S1
PA = P.mean(dim=('lat','lon')).groupby('time.year').mean('time')
sort_index = PA.sortby(PA)
dryS1 = sort_index[0:5]['year']
wetS1 = sort_index[-5:]['year']
print('Dry years S1',dryS1['year'].values)
print('Wet years S1',wetS1['year'].values)
PW1 = P.sel(time=P['time.year'].isin(wetS1['year'].values)).groupby('time.dayofyear').mean(dim=('time','lat','lon'))
PD1 = P.sel(time=P['time.year'].isin(dryS1['year'].values)).groupby('time.dayofyear').mean(dim=('time','lat','lon'))
print('----------------------------------------------------')


#Season timing
power_ratio,on1,ce1,on2,ce2,on3,ce3,on1_years,ce1_years,on2_years,ce2_years,on3_years,ce3_years = ocd.xarray_on_cess_point(P.mean(dim=("lat","lon")))

mean_onset_1 = on1_years.mean('year').values 
mean_cess_1 = ce1_years.mean('year').values 
mean_onset_2 = on2_years.mean('year').values 
mean_cess_2 = ce2_years.mean('year').values
print('All years mean onset 1', mean_onset_1, 'All years mean cessation 1', mean_cess_1)
print('All years mean onset 2', mean_onset_2, 'All years mean cessation 2', mean_cess_2)
print('----------------------------------------------------')

#Onset and cessation of wet and dry composite years

print('Wet years onset 1: ',on1_years.sel(year=on1_years['year'].isin(wetS1['year'].values)).mean('year').values ,
      'Dry years onset 1: ',on1_years.sel(year=on1_years['year'].isin(dryS1['year'].values)).mean('year').values)
print('Wet years cessation 1: ',ce1_years.sel(year=ce1_years['year'].isin(wetS1['year'].values)).mean('year').values ,
      'Dry years cessation1: ',ce1_years.sel(year=ce1_years['year'].isin(dryS1['year'].values)).mean('year').values)
print('----------------------------------------------------')

#Onset and cessation of early and late composite years

sort_index_onset = on1_years.sortby(on1_years)
early_onset1 = sort_index_onset[0:5]
late_onset1 = sort_index_onset[-5:]
early_onset_precip = PA.where(PA['year'].isin(early_onset1['year']),drop=True)
late_onset_precip = PA.where(PA['year'].isin(late_onset1['year']),drop=True)
print('Mean early onset comp S1 ',early_onset1.mean('year').values, 'Years: ',early_onset1['year'].values, 'Early onset comp S1 ',early_onset1.values)
print('Mean late onset comp S1 ',late_onset1.mean('year').values, 'Years: ',late_onset1['year'].values, 'Late onset comp S1 ',late_onset1.values)

sort_index_cess = ce1_years.sortby(ce1_years)
early_cess1 = sort_index_cess[0:5]
late_cess1 = sort_index_cess[-5:]
early_cess_precip = PA.where(PA['year'].isin(early_cess1['year']),drop=True)
late_cess_precip = PA.where(PA['year'].isin(late_cess1['year']),drop=True)
print('Mean early cess comp S1 ',early_cess1.mean('year').values, 'Years: ',early_cess1['year'].values, 'Early cess comp S1 ',early_cess1.values)
print('Mean late cess comp S1 ',late_cess1.mean('year').values, 'Years: ',late_cess1['year'].values, 'Late cess comp S1 ',late_cess1.values)
print('----------------------------------------------------')

#Plot everything - SEASON 1

fig,ax = plt.subplots(figsize=(13, 6))
Pdays = P.mean(dim = ('lat','lon')).groupby('time.dayofyear').mean(dim = ('time'))
Pplot = Pdays.rolling(dayofyear=20, center=True).mean('dayofyear')
Pplot.plot(linewidth=3,label='Precip Climatology')

plt.axvline(x=on1_years.mean('year').values, color='firebrick', linestyle='-', linewidth=5,alpha=0.5,label='Mean Onset')
plt.axvline(x=early_onset1.mean('year'), color='firebrick', linestyle='-.', linewidth=2,alpha=0.5,label='Early Onset')
plt.axvline(x=late_onset1.mean('year'), color='firebrick', linestyle='--', linewidth=2,alpha=0.5,label='Late Onset')

plt.axvline(x=ce1_years.mean('year').values, color='green', linestyle='-', linewidth=5,alpha=0.5,label='Mean Cessation')
plt.axvline(x=early_cess1.mean('year'), color='green', linestyle='-.', linewidth=2,alpha=0.5,label='Early Cessation')
plt.axvline(x=late_cess1.mean('year'), color='green', linestyle='--', linewidth=2,alpha=0.5,label='Late Cessation')

plt.axvline(x=on1_years.sel(year=on1_years['year'].isin(wetS1['year'].values)).mean('year').values, color='purple', linestyle='--', linewidth=2,alpha=1,label='Wet years Mean Onset')
plt.axvline(x=on1_years.sel(year=on1_years['year'].isin(dryS1['year'].values)).mean('year').values, color='orange', linestyle='--', linewidth=2,alpha=1,label='Dry years Mean Onset')
plt.axvline(x=ce1_years.sel(year=ce1_years['year'].isin(wetS1['year'].values)).mean('year').values, color='purple', linestyle='dotted', linewidth=2,alpha=1,label='Wet years Mean Cessation')
plt.axvline(x=ce1_years.sel(year=ce1_years['year'].isin(dryS1['year'].values)).mean('year').values, color='orange', linestyle='dotted', linewidth=2,alpha=1,label='Dry years Mean Cessation')

PonE = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(early_onset1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PonL = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(late_onset1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')

PceE = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(early_cess1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')
PceL = P.mean(dim=('lat','lon')).sel(time=P['time.year'].isin(late_cess1['year'].values)).groupby('time.dayofyear').mean('time').rolling(dayofyear=30, center=True).mean('dayofyear')

#plt.fill_between(PonE[0:365].dayofyear,PonE[0:365],PonL[0:365], color='red',alpha=0.3,label="Early vs Late Onset")
#plt.fill_between(PceL[0:365].dayofyear,PceE[0:365],PceL[0:365],color='green',alpha=0.3,label="Early vs Late Cess")
#PonE.plot(linestyle='--',color='red',alpha=1,label="Early Onset Clim")
#PceL.plot(linestyle='--',color='green',alpha=1,label="Late Cessation Clim")

PW1 = PW1.rolling(dayofyear=20, center=True).mean('dayofyear')
PD1 = PD1.rolling(dayofyear=20, center=True).mean('dayofyear')
PW1.plot(linestyle='-',color='purple',alpha=1,label="Wet Composite Clim")
PD1.plot(linestyle='-',color='orange',alpha=1,label="Dry Composite Clim")

ax2 = ax.twinx()
ax3 = ax.twinx()
ax3.spines.right.set_position(("axes", 1.08))

ax2.plot(rho_mon['S']['dayofyear'],rho_mon['S'],linestyle='-',linewidth=2,color='darkgreen',alpha=0.3)
ax2.scatter(rho_mon['S']['dayofyear'],rho_mon['S'],s=70,color='darkgreen',label='Recycling Ratio',alpha=0.8)
ax2.set(ylim=(0,0.4),ylabel='Recycling rho')

FPR = FP.rolling(dayofyear=3, center=True).mean('dayofyear')
ax3.plot(FPR['dayofyear'],FPR,linestyle='-',linewidth=3,color='teal',alpha=0.7,label='FAPAR')
ax3.set(ylim=(0,0.85),ylabel='Fraction of Absorbed \n Photosynthetically Active Radiation')

ax.legend(loc='best')
ax2.legend(loc=1)
ax3.legend(loc=2)
plt.title('Southern Band')
plt.savefig(datap+'S_S1_timing_pres.png',dpi=200,transparent=True,bbox_inches='tight')
plt.show()
plt.clf()
